### Silbentrennung – Lösung Schritt für Schritt

*Jugendwettbewerb Informatik 2025, 3. Runde, Junioraufgabe 2 (= 44. Bundeswettbewerb Informatik, 1. Runde, Junioraufgabe 2)*

Die Silbentrennung in heutigen Textverarbeitungsprogrammen ist recht gut, enthält aber meist noch einige Fehler.
Lars möchte wissen, ob er selbst ein ähnlich gutes oder besseres Programm erstellen kann.

Er überlegt sich einige Regeln, mit denen er beginnen möchte:

- Ein Wort kann zwischen zwei Konsonanten getrennt werden.
- Erster und letzter Buchstabe eines Wortes dürfen nicht abgetrennt werden.
- Bei drei aufeinanderfolgenden Konsonanten kann nur nach dem ersten Konsonanten getrennt werden.
- Ein Wort kann nach einem Vokal getrennt werden, falls nicht zwei Konsonanten folgen.

Lars merkt, dass sich Regeln gelegentlich widersprechen. Er wendet dann die letzte davon an.
Lars ist mit den Ergebnissen dieser Silbentrennung schon recht zufrieden, aber Wörter wie z.B. „Sauerkraut“ werden doch
recht merkwürdig getrennt. Hast du Ideen für weitere Regeln, die Lars' Programm verbessern?

#### Junioraufgabe 2

Erstelle ein Silbentrennungsprogramm, welches Lars' Regeln und gegebenenfalls deine eigenen umsetzt, und teste es an den
Texten auf den BWINF-Webseiten. Es soll in jedem Wort die möglichen Trennstellen markieren. Bewerte die Leistung deines
Programms.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest,
und dokumentiere die Ergebnisse.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `silben01.txt` bis `silben08.txt` und `silben0a.txt` („Informatik“, eine zusätzliche
eigene Datei). Jede Datei enthält einen kurzen Text. Die README nennt das Ergebnis von Lars' Regeln für `silben05.txt`:

> Se-i-n Va-te-r ist Ba-u-e-r und er-nte-t Ma-i-s.

Das bestätigt die wörtliche Auslegung „die letzte Regel gewinnt“: Regel 4 überstimmt Regel 2, daher „Se-i-n“ und „Ma-i-s“.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Regelsysteme programmieren:** Die Regeln werden der Reihe nach geprüft, und die **letzte zutreffende** Regel gewinnt.
  Dafür genügt eine Variable `erlaubt` und für jede Regel ein eigenes `if` (kein `elif`).
- **Nachbarn ansehen:** An jeder Stelle im Wort betrachtet man den Buchstaben davor und die danach. Dabei muss man aufpassen, dass
  es diese Buchstaben am Wortanfang und am Wortende nicht gibt (sonst `IndexError`).
- **Text in Wörter zerlegen:** Den Text Zeichen für Zeichen durchgehen und die Buchstaben zu Wörtern sammeln.
- **Leistung messen:** Das Ergebnis mit einer Referenz vergleichen und richtig, falsch und fehlend zählen.
  So kann man verschiedene Versionen des Programms vergleichen.
- **Schrittweise verbessern:** Fehler ansehen, eine Regel dazu überlegen, neu messen.

**Nützliche Funktionen und Bibliotheken**
- `zeichen.isalpha()` (ist es ein Buchstabe?), `lower()`, `in`, Ausschnitte wie `liste[:p]` und `liste[p:]`,
  `''.join(liste)` und f-Strings wie `{text:35}` für ausgerichtete Ausgaben.
- Für richtige Silbentrennung gibt es Programme mit Wörterbüchern. Das verbreitete Verfahren von Liang (aus dem Textsatzsystem
  TeX) benutzt Tabellen mit Trennmustern. In Python gibt es dafür die Bibliothek **Pyphen**. Hier schreiben wir
  alles selbst und brauchen keine Bibliothek.

## Die Idee

Für jede Stelle `p` zwischen zwei Buchstaben eines Wortes (die Stelle `p` liegt vor dem Buchstaben `wort[p]`) fragen wir alle
vier Regeln der Reihe nach. Jede Regel, die zutrifft, setzt die Antwort auf „trennen“ oder „nicht trennen“. Weil die Regeln
in dieser Reihenfolge geprüft werden, gewinnt am Ende die letzte zutreffende Regel.

| Regel | Bedingung an der Stelle `p` | Antwort |
|:-:|---|---|
| 1 | links und rechts steht ein Konsonant | trennen |
| 2 | `p` ist die Stelle nach dem ersten oder vor dem letzten Buchstaben | nicht trennen |
| 3 | die drei Buchstaben `wort[p-2]`, `wort[p-1]`, `wort[p]` sind Konsonanten | nicht trennen |
| 4 | links steht ein Vokal, und es folgen nicht zwei Konsonanten | trennen |

Zu Regel 3: Bei drei Konsonanten hintereinander darf nur nach dem ersten getrennt werden. Also ist die Stelle nach dem
**zweiten** Konsonanten verboten, und genau das drückt die Bedingung aus. Bei mehr als drei Konsonanten gilt die Regel für jede
Dreiergruppe.

**Beispiel „gerne“:** Stelle 1 (g|e) verbietet Regel 2. Stelle 2 (e|r): Links ein Vokal, rechts folgen zwei Konsonanten
(`r`, `n`), also trifft Regel 4 nicht zu. Stelle 3 (r|n): zwei Konsonanten, Regel 1 erlaubt, kein anderes Verbot, also wird
getrennt. Stelle 4 (n|e): Regel 2 verbietet. Ergebnis: **ger-ne**.

## Schritt 1: Vokale und Konsonanten

Vokale sind `a e i o u` und die Umlaute `ä ö ü`. Alle anderen Buchstaben zählen wir zu den Konsonanten (auch `ß` und `y`).
Das `y` ist eigentlich ein Grenzfall: In „Hobbyte“ klingt es wie ein Vokal, in „Yacht“ wie ein Konsonant.
Wir zählen es zunächst zu den Konsonanten. Mit `lower()` werden Großbuchstaben genauso erkannt.

Die Funktion schaut nur auf den **ersten** Buchstaben des übergebenen Textes. Für einen einzelnen Buchstaben ist das
dieser Buchstabe selbst. Später geben wir ihr auch Buchstabengruppen wie `'sch'` oder `'au'`, und dann zählt der erste Buchstabe.

In [ ]:
VOKALE = 'aeiouäöü'


def ist_vokal(teil):
    return teil[0].lower() in VOKALE


def ist_konsonant(teil):
    return not ist_vokal(teil)


for b in ['a', 'E', 'ü', 'k', 'S', 'ß', 'y']:
    print(b, 'Vokal' if ist_vokal(b) else 'Konsonant')

a Vokal
E Vokal
ü Vokal
k Konsonant
S Konsonant
ß Konsonant
y Konsonant


## Schritt 2: Die Trennstellen nach Lars' Regeln

Die Funktion `trennstellen_lars(wort)` gibt eine Liste aller Stellen zurück, an denen getrennt werden darf. Für jede Stelle `p`
von 1 bis `n - 1` (`n` ist die Länge des Wortes) setzen wir `erlaubt` am Anfang auf `False` und prüfen dann die vier Regeln
in ihrer Reihenfolge.

Zwei Stellen verdienen Aufmerksamkeit:

- **Regel 3** braucht `wort[p - 2]`. Das gibt es erst ab `p >= 2`.
- **Regel 4** braucht `wort[p + 1]`. Das gibt es nur, wenn `p + 1 < n` ist. Die Prüfung `p + 1 < n` steht in der Bedingung **vor**
  `wort[p + 1]`: Python hört bei `and` auf, sobald ein Teil `False` ist, und liest dann `wort[p + 1]` gar nicht erst.

Die Funktion liest `wort` nur über `wort[...]` und `len(wort)`. Deshalb funktioniert sie später auch mit einer Liste von
Buchstabengruppen.

In [ ]:
def trennstellen_lars(wort):
    n = len(wort)
    stellen = []
    for p in range(1, n):
        erlaubt = False

        # Regel 1: zwischen zwei Konsonanten darf getrennt werden
        if ist_konsonant(wort[p - 1]) and ist_konsonant(wort[p]):
            erlaubt = True

        # Regel 2: nach dem ersten und vor dem letzten Buchstaben nicht trennen
        if p == 1 or p == n - 1:
            erlaubt = False

        # Regel 3: bei drei Konsonanten hintereinander nicht nach dem zweiten trennen
        if p >= 2 and ist_konsonant(wort[p - 2]) and ist_konsonant(wort[p - 1]) and ist_konsonant(wort[p]):
            erlaubt = False

        # Regel 4: nach einem Vokal darf getrennt werden, wenn nicht zwei Konsonanten folgen
        if ist_vokal(wort[p - 1]) and not (p + 1 < n and ist_konsonant(wort[p]) and ist_konsonant(wort[p + 1])):
            erlaubt = True

        if erlaubt:
            stellen.append(p)
    return stellen


for wort in ['gerne', 'Sauerkraut', 'Karpfen', 'Es', 'Dampfschiff']:
    print(wort, trennstellen_lars(wort))

gerne [3]
Sauerkraut [2, 3, 5, 8, 9]
Karpfen [3, 6]
Es [1]
Dampfschiff [3]


## Schritt 3: Trennstellen markieren

`markiere(wort, stellen)` baut ein neues Wort und fügt an jeder Trennstelle einen Bindestrich ein.
Die Stelle `i` liegt **vor** dem Buchstaben `wort[i]`: Ist `i` eine Trennstelle, kommt zuerst der Bindestrich,
dann der Buchstabe.

In [ ]:
def markiere(wort, stellen):
    ergebnis = ''
    for i in range(len(wort)):
        if i in stellen:
            ergebnis = ergebnis + '-'
        ergebnis = ergebnis + wort[i]
    return ergebnis


def trenne_lars(wort):
    return markiere(wort, trennstellen_lars(wort))


for wort in ['gerne', 'Sauerkraut', 'Karpfen', 'Es', 'Dampfschiff']:
    print(trenne_lars(wort))

ger-ne
Sa-u-er-kra-u-t
Kar-pfe-n
E-s
Dam-pfschiff


Das sind genau die Ergebnisse, die man nach Lars' Regeln erwartet. Auch die auffälligen Trennungen sind „richtig“, denn sie
folgen aus den Regeln: „E-s“ (Regel 4 schlägt Regel 2) und „Sa-u-er-kra-u-t“.

## Schritt 4: Ganze Texte trennen

Die Dateien enthalten ganze Sätze mit Satzzeichen. Wir gehen den Text Zeichen für Zeichen durch und sammeln Buchstaben in
der Variablen `wort`. Kommt ein Zeichen, das kein Buchstabe ist, ist das Wort zu Ende: Wir hängen das getrennte Wort und danach
das Zeichen an das Ergebnis an. Satzzeichen und Leerzeichen bleiben so unverändert. Nach der Schleife kann noch ein Wort übrig
sein (wenn der Text mit einem Buchstaben endet), das hängen wir auch noch an.

Damit wir später verschiedene Versionen des Programms ausprobieren können, ruft `trenne_text` die Funktion `trenne(wort)`
auf. Diese steht immer für die **aktuelle Version**. Im Moment ist das Lars' Version. Wenn wir später eine bessere Version
haben, definieren wir `trenne` einfach neu, und alles andere funktioniert unverändert.

In [ ]:
def trenne(wort):
    return trenne_lars(wort)


def trenne_text(text):
    ergebnis = ''
    wort = ''
    for zeichen in text:
        if zeichen.isalpha():
            wort = wort + zeichen
        else:
            if wort != '':
                ergebnis = ergebnis + trenne(wort)
                wort = ''
            ergebnis = ergebnis + zeichen
    if wort != '':
        ergebnis = ergebnis + trenne(wort)
    return ergebnis


def lies_text(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        return f.read().strip()


print(trenne_text('Mein Name ist Lars und ich esse schrecklich gerne Sauerkraut.'))

Me-i-n Na-me ist Lars und ich es-se schrec-klich ger-ne Sa-u-er-kra-u-t.


Jetzt lassen wir Lars' Regeln auf alle Beispieltexte los. Für `silben05.txt` steht das erwartete Ergebnis in der README
(„Se-i-n Va-te-r ist Ba-u-e-r und er-nte-t Ma-i-s.“).

In [ ]:
DATEIEN = ['1', '2', '3', '4', '5', '6', '7', '8', 'a']


def trenne_alle():
    for nr in DATEIEN:
        dateiname = f'beispieldaten/silben0{nr}.txt'
        print(f'{dateiname}:')
        print(trenne_text(lies_text(dateiname)))
        print()


trenne_alle()

beispieldaten/silben01.txt:
Me-i-n Na-me ist Lars und ich es-se schrec-klich ger-ne Sa-u-er-kra-u-t.

beispieldaten/silben02.txt:
De-r Ka-pi-tä-n ist e-ben-fal-ls Prä-si-dent de-r ör-tlic-he-n Dam-pfschif-ffah-rtsge-sel-lschaft.

beispieldaten/silben03.txt:
E-s ist ar-schkalt!

beispieldaten/silben04.txt:
Ist da-s A-u-di-o-sig-na-l i-m Ra-di-o schlec-ht?

beispieldaten/silben05.txt:
Se-i-n Va-te-r ist Ba-u-e-r und er-nte-t Ma-i-s.

beispieldaten/silben06.txt:
Ich an-gle Kar-pfe-n.

beispieldaten/silben07.txt:
Wa-s sind ac-ht Hob-bits? E-i-n Hob-byte!

beispieldaten/silben08.txt:
Fre-u-de, schö-ne-r Göt-ter-fun-ke-n, Toc-hte-r a-u-s E-li-si-u-m, Wi-r bet-re-te-n fe-u-er-trun-ke-n, Him-mlis-che, de-i-n He-i-lig-thu-m. De-i-ne Za-u-be-r bin-de-n wi-e-de-r, Wa-s di-e Mo-de streng get-he-ilt, Al-le Men-sche-n wer-de-n Brü-de-r, Wo de-i-n san-fte-r Flü-ge-l we-ilt.

beispieldaten/silben0a.txt:
In-for-ma-ti-k



## Schritt 5: Bewerten, wie gut die Trennung ist

Die Aufgabe verlangt, dass wir die Leistung bewerten. „Sieht ganz gut aus“ reicht nicht, wir wollen eine Zahl,
mit der man Versionen vergleichen kann. Dafür brauchen wir die **richtige Trennung** der Wörter (nach dem Duden).

Die Liste `REFERENZ` enthält alle Wörter der Beispieltexte, jedes nur einmal, mit der Trennung nach heutiger Rechtschreibung.
Drei Anmerkungen:

- Bei manchen Wörtern erlaubt der Duden zwei Trennungen (etwa „Au-dio-si-gnal“ und „Au-dio-sig-nal“). Hier steht nur eine.
- `silben08.txt` ist ein Gedicht von Schiller in alter Rechtschreibung („Heiligthum“, „getheilt“). Die Wörter sind so getrennt,
  wie man sie heute trennen würde.
- „Hobbyte“ ist ein Wortspiel (acht Bits sind ein Byte, acht Hobbits ein „Hobbyte“) und wird wie „Byte“ getrennt.

In [ ]:
REFERENZ = """
Mein Na-me ist Lars und ich es-se schreck-lich ger-ne Sau-er-kraut
Der Ka-pi-tän eben-falls Prä-si-dent der ört-li-chen Dampf-schiff-fahrts-ge-sell-schaft
Es arsch-kalt
Ist das Au-dio-sig-nal im Ra-dio schlecht
Sein Va-ter Bau-er ern-tet Mais
Ich ang-le Karp-fen
Was sind acht Hob-bits Ein Hob-byte
Freu-de schö-ner Göt-ter-fun-ken Toch-ter aus Eli-si-um Wir be-tre-ten feu-er-trun-ken Himm-li-sche dein
Hei-lig-thum Dei-ne Zau-ber bin-den wie-der die Mo-de streng ge-theilt Al-le Men-schen wer-den Brü-der
Wo sanf-ter Flü-gel weilt
In-for-ma-tik
""".split()
print(len(REFERENZ), 'Wörter')

68 Wörter


Zum Vergleichen brauchen wir für ein getrenntes Wort wie `'Sau-er-kraut'` die Liste seiner Trennstellen (hier `[3, 5]`,
wie bei Lars: Stelle 3 liegt vor dem vierten Buchstaben). `stellen_aus` zählt dazu die Buchstaben und merkt sich bei jedem
Bindestrich, wie viele es bisher waren.

`bewerte(name)` trennt dann jedes Wort der Referenz mit der aktuellen Version `trenne` und zählt:

- **richtig:** Stellen, die das Programm findet und die auch in der Referenz stehen,
- **falsch:** Stellen, an denen das Programm trennt, obwohl man dort nicht trennen darf,
- **fehlend:** richtige Stellen, die das Programm nicht findet.

Falsche Trennungen sind schlimmer als fehlende: Ein Wort wie „Sa-u-er-kraut“ am Zeilenende sieht falsch aus. Fehlt eine
Trennstelle, wird das Wort eben an einer anderen Stelle oder gar nicht getrennt, das fällt kaum auf.

In [ ]:
def stellen_aus(getrennt):
    stellen = []
    buchstaben = 0
    for zeichen in getrennt:
        if zeichen == '-':
            stellen.append(buchstaben)
        else:
            buchstaben = buchstaben + 1
    return stellen


def bewerte(name):
    richtig = 0
    falsch = 0
    fehlend = 0
    for wort_richtig in REFERENZ:
        wort = wort_richtig.replace('-', '')            # das Wort ohne Bindestriche
        programm = stellen_aus(trenne(wort))            # Trennstellen des Programms
        korrekt = stellen_aus(wort_richtig)             # richtige Trennstellen
        for s in programm:
            if s in korrekt:
                richtig = richtig + 1
            else:
                falsch = falsch + 1
        for s in korrekt:
            if s not in programm:
                fehlend = fehlend + 1
    print(f'{name:36} richtig: {richtig:2}  falsch: {falsch:2}  fehlend: {fehlend:2}')


bewerte("Lars' Regeln")

Lars' Regeln                         richtig: 45  falsch: 80  fehlend: 17


Die Referenz hat 62 richtige Trennstellen. Lars findet davon 45, trennt aber 80-mal falsch. Das ist die Zahl, die wir verbessern wollen.

Damit man sieht, **was** falsch ist, schreiben wir noch eine Funktion `fehler()`. Sie gibt alle Wörter aus, bei denen die Trennung nicht mit der
Referenz übereinstimmt. In einem f-String füllt `{getrennt:35}` den Text mit Leerzeichen auf 35 Zeichen auf, so stehen die richtigen Trennungen untereinander.

In [ ]:
def fehler():
    for wort_richtig in REFERENZ:
        wort = wort_richtig.replace('-', '')
        getrennt = trenne(wort)
        if getrennt != wort_richtig:
            print(f'{getrennt:35} richtig: {wort_richtig}')


fehler()

Me-i-n                              richtig: Mein
schrec-klich                        richtig: schreck-lich
Sa-u-er-kra-u-t                     richtig: Sau-er-kraut
De-r                                richtig: Der
Ka-pi-tä-n                          richtig: Ka-pi-tän
e-ben-fal-ls                        richtig: eben-falls
de-r                                richtig: der
ör-tlic-he-n                        richtig: ört-li-chen
Dam-pfschif-ffah-rtsge-sel-lschaft  richtig: Dampf-schiff-fahrts-ge-sell-schaft
E-s                                 richtig: Es
ar-schkalt                          richtig: arsch-kalt
da-s                                richtig: das
A-u-di-o-sig-na-l                   richtig: Au-dio-sig-nal
i-m                                 richtig: im
Ra-di-o                             richtig: Ra-dio
schlec-ht                           richtig: schlecht
Se-i-n                              richtig: Sein
Va-te-r                             richtig: Va-ter
Ba-u-e-r           

Man erkennt drei Gruppen von Fehlern:

1. **Einzelne Buchstaben am Rand** werden abgetrennt: E-s, de-r, Kar-pfe-n. Regel 4 schlägt Regel 2.
2. **Buchstaben, die zusammengehören**, werden getrennt: `ei` in Me-i-n, `au` in Sa-u-er, `ch` in Toc-hte-r, `ck` in schrec-klich.
3. **Mehrere Konsonanten** werden an der falschen Stelle getrennt: Kar-pfen statt Karp-fen. Im Deutschen kommt von mehreren
   Konsonanten nur der letzte in die neue Silbe, Lars macht es umgekehrt.

Dazu kommen Zusammensetzungen und Fremdwörter, die man mit Regeln über Vokale und Konsonanten nicht lösen kann.

## Schritt 6: Verbesserung 1 – Regel 2 gewinnt immer

Gruppe 1 ist leicht zu beheben: Der erste und der letzte Buchstabe sollen nie abgetrennt werden, egal was die anderen Regeln sagen.
Wir nehmen die Trennstellen von Lars und werfen die Stellen 1 und `n - 1` einfach weg. Dann ist Regel 2 praktisch die letzte Regel.

Die Definition von `trenne` macht diese Version zur aktuellen. Deshalb bewertet `bewerte` danach die neue Version.

In [ ]:
def trenne_v1(wort):
    stellen = []
    for p in trennstellen_lars(wort):
        if p != 1 and p != len(wort) - 1:
            stellen.append(p)
    return markiere(wort, stellen)


def trenne(wort):
    return trenne_v1(wort)


print(trenne_v1('Es'), trenne_v1('Sauerkraut'), trenne_v1('Informatik'))
bewerte('Version 1: Rand nie abtrennen')

Es Sa-u-er-kra-ut In-for-ma-tik
Version 1: Rand nie abtrennen        richtig: 45  falsch: 36  fehlend: 17


Die falschen Trennungen sinken von 80 auf 36, die richtigen bleiben gleich. Eine einfache und gute Verbesserung.

## Schritt 7: Verbesserung 2 – Buchstabengruppen zusammenhalten

Gruppe 2: Manche Laute schreibt man mit mehreren Buchstaben, und die darf man nicht trennen. Wir halten diese Gruppen zusammen:

- Konsonanten: `sch`, `ch`, `ck`, `ph`, `th`
- Vokale (Doppellaute): `au`, `äu`, `eu`, `ei`, `ai`, `ie`

Die Idee: Wir zerlegen das Wort zuerst in **Einheiten**, zum Beispiel „Sauerkraut“ in `['S', 'au', 'e', 'r', 'k', 'r', 'au', 't']`.
Jede Einheit zählt dann als **ein** Vokal oder Konsonant, je nach ihrem ersten Buchstaben (deshalb schaut `ist_vokal` nur auf
den ersten Buchstaben). Dann wenden wir die Regeln auf die Liste der Einheiten an, und innerhalb einer Gruppe wird nie getrennt.

Die Funktion `einheiten` geht mit einer `while`-Schleife durch das Wort. An jeder Position `i` prüft sie, ob dort eine der Gruppen beginnt:
`wort[i:i + len(gruppe)]` sind die nächsten Buchstaben in der Länge der Gruppe. Ist es eine Gruppe, wird sie als Ganzes übernommen
und `i` springt entsprechend weiter, sonst wird nur ein einzelner Buchstabe übernommen. Die längeren Gruppen (`sch`) stehen vor den
kürzeren (`ch`), damit `sch` nicht als `s` + `ch` zerlegt wird.

In [ ]:
GRUPPEN = ['sch', 'ch', 'ck', 'ph', 'th', 'au', 'äu', 'eu', 'ei', 'ai', 'ie']


def einheiten(wort):
    teile = []
    i = 0
    while i < len(wort):
        gefunden = False
        for gruppe in GRUPPEN:
            if not gefunden and wort[i:i + len(gruppe)].lower() == gruppe:
                teile.append(wort[i:i + len(gruppe)])
                i = i + len(gruppe)
                gefunden = True
        if not gefunden:
            teile.append(wort[i])
            i = i + 1
    return teile


print(einheiten('Sauerkraut'))
print(einheiten('Schrecklich'))
print(einheiten('Heiligthum'))

['S', 'au', 'e', 'r', 'k', 'r', 'au', 't']
['Sch', 'r', 'e', 'ck', 'l', 'i', 'ch']
['H', 'ei', 'l', 'i', 'g', 'th', 'u', 'm']


Die Trennstellen von Lars funktionieren auch für Einheiten, weil `trennstellen_lars` nur `wort[p]` und `len(wort)` benutzt
(und `ist_vokal` den ersten Buchstaben der Einheit ansieht). Nur Regel 2 passt nicht mehr: Sie spricht von **Buchstaben**.
Bei „Audio“, das aus `['Au', 'd', 'i', 'o']` besteht, ist Stelle 1 die Stelle nach `Au`, und dort darf getrennt werden („Au-dio“).
Regel 2 muss also lauten: Links und rechts der Trennstelle müssen mindestens **zwei Buchstaben** stehen.
`''.join(teile[:p])` hängt die Einheiten links der Stelle zu einem Text zusammen, `''.join(teile[p:])` die rechts davon.

In [ ]:
def trennstellen_v2(teile):
    stellen = []
    for p in trennstellen_lars(teile):
        links = ''.join(teile[:p])
        rechts = ''.join(teile[p:])
        if len(links) > 1 and len(rechts) > 1:
            stellen.append(p)
    return stellen


def trenne_v2(wort):
    teile = einheiten(wort)
    return markiere(teile, trennstellen_v2(teile))


def trenne(wort):
    return trenne_v2(wort)


print(trenne_v2('Sauerkraut'), trenne_v2('schrecklich'), trenne_v2('Audiosignal'), trenne_v2('Mein'))
bewerte('Version 2: Buchstabengruppen')

Sau-er-kraut schreck-li-ch Au-di-o-sig-nal Mein
Version 2: Buchstabengruppen         richtig: 50  falsch: 15  fehlend: 12


„Sauerkraut“ ist jetzt richtig, und die falschen Trennungen sinken auf 15. Aber es gibt einen neuen Fehler: **schreck-li-ch**.
Die Einheit `ch` am Ende hat zwei Buchstaben, deshalb lässt die neue Regel 2 die Trennung davor durch.
Das beheben wir im nächsten Schritt.

## Schritt 8: Verbesserung 3 – Trennen wie im Duden

Gruppe 3: Im Deutschen gilt (Duden): **Von mehreren Konsonanten kommt nur der letzte in die neue Zeile**
(Karp-fen, Him-mel, sanf-ter). In der Sprache der Einheiten: Man trennt zwischen zwei Konsonanten, wenn nach dem zweiten
ein Vokal folgt. Dazu kommt noch eine einfache Idee: **Jede Silbe braucht einen Vokal.** Links und rechts der Trennstelle muss
also mindestens ein Vokal stehen. Das behebt auch „schreck-li-ch“ und verhindert „st-reng“.

Die neuen Regeln (wieder in dieser Reihenfolge, die letzte gewinnt):

- **Regel A** (neu, ersetzt Regeln 1 und 3): zwischen zwei Konsonanten, wenn nach dem zweiten ein Vokal folgt,
- **Regel 4** von Lars: nach einem Vokal, falls nicht zwei Konsonanten folgen,
- **Regel B** (neu): links und rechts der Stelle muss ein Vokal stehen,
- **Regel 2** wie in Schritt 7: links und rechts mindestens zwei Buchstaben.

Für Regel B hilft `hat_vokal(teile)`. Sie gibt `True` zurück, sobald eine Einheit ein Vokal ist, und `False` erst nach der
Schleife, wenn keine gefunden wurde. Ein `return` mitten in einer Schleife beendet die Funktion sofort.

In [ ]:
def hat_vokal(teile):
    for teil in teile:
        if ist_vokal(teil):
            return True
    return False


def trennstellen_v3(teile):
    n = len(teile)
    stellen = []
    for p in range(1, n):
        erlaubt = False

        # Regel A: von mehreren Konsonanten kommt nur der letzte in die neue Silbe
        if ist_konsonant(teile[p - 1]) and ist_konsonant(teile[p]) and p + 1 < n and ist_vokal(teile[p + 1]):
            erlaubt = True

        # Regel 4: nach einem Vokal, falls nicht zwei Konsonanten folgen
        if ist_vokal(teile[p - 1]) and not (p + 1 < n and ist_konsonant(teile[p]) and ist_konsonant(teile[p + 1])):
            erlaubt = True

        # Regel B: jede Silbe braucht einen Vokal
        if not hat_vokal(teile[:p]) or not hat_vokal(teile[p:]):
            erlaubt = False

        # Regel 2: keine einzelnen Buchstaben abtrennen
        if len(''.join(teile[:p])) == 1 or len(''.join(teile[p:])) == 1:
            erlaubt = False

        if erlaubt:
            stellen.append(p)
    return stellen


def trenne_v3(wort):
    teile = einheiten(wort)
    return markiere(teile, trennstellen_v3(teile))


def trenne(wort):
    return trenne_v3(wort)


print(trenne_v3('Karpfen'), trenne_v3('Dampfschifffahrtsgesellschaft'), trenne_v3('streng'))
bewerte('Version 3: Duden-Regel und Vokal')

Karp-fen Dampf-schiff-fahrts-ge-sell-schaft streng
Version 3: Duden-Regel und Vokal     richtig: 58  falsch:  5  fehlend:  4


Ein großer Sprung: Nur noch 5 falsche Trennungen. Das Bandwurmwort „Dampfschifffahrtsgesellschaft“ wird jetzt richtig getrennt.
Welche Fehler bleiben übrig?

In [ ]:
fehler()

Sau-erk-raut                        richtig: Sau-er-kraut
Au-di-o-sig-nal                     richtig: Au-dio-sig-nal
Hobby-te                            richtig: Hob-byte
bet-re-ten                          richtig: be-tre-ten
feu-ert-run-ken                     richtig: feu-er-trun-ken


Die Fehler haben alle mit dem **Wortschatz** zu tun, nicht mit Vokalen und Konsonanten:

- **Sauerkraut, feuertrunken:** zusammengesetzte Wörter (Sauer|kraut, feuer|trunken). Die Grenze zwischen den Wortteilen geht vor.
- **betreten:** Vorsilbe be-, und `tr` gehört zusammen an den Anfang einer Silbe.
- **Audiosignal:** Ein Fremdwort. In „Au-dio“ (wie in „Ra-dio“) trennt man `i` und `o` nicht, in „Eli-si-um“ dagegen schon.
- **Hobbyte:** ein Wortspiel mit dem englischen „Byte“. Das kann keine Regel wissen.

Die echten Textprogramme benutzen dafür ein **Wörterbuch** mit Wortteilen und Ausnahmen.

## Schritt 9: Alle Texte mit der besten Version

Hier ist das Ergebnis für alle Beispieltexte. Die aktuelle Version `trenne` ist `trenne_v3`.

In [ ]:
trenne_alle()

beispieldaten/silben01.txt:
Mein Na-me ist Lars und ich es-se schreck-lich ger-ne Sau-erk-raut.

beispieldaten/silben02.txt:
Der Ka-pi-tän ist eben-falls Prä-si-dent der ört-li-chen Dampf-schiff-fahrts-ge-sell-schaft.

beispieldaten/silben03.txt:
Es ist arsch-kalt!

beispieldaten/silben04.txt:
Ist das Au-di-o-sig-nal im Ra-dio schlecht?

beispieldaten/silben05.txt:
Sein Va-ter ist Bau-er und ern-tet Mais.

beispieldaten/silben06.txt:
Ich ang-le Karp-fen.

beispieldaten/silben07.txt:
Was sind acht Hob-bits? Ein Hobby-te!

beispieldaten/silben08.txt:
Freu-de, schö-ner Göt-ter-fun-ken, Toch-ter aus Eli-si-um, Wir bet-re-ten feu-ert-run-ken, Himm-li-sche, dein Hei-lig-thum. Dei-ne Zau-ber bin-den wie-der, Was die Mo-de streng ge-theilt, Al-le Men-schen wer-den Brü-der, Wo dein sanf-ter Flü-gel weilt.

beispieldaten/silben0a.txt:
In-for-ma-tik



## Für Neugierige: Typische Silbenanfänge

Bei „Sauer**kr**aut“, „be**tr**eten“ und „feuer**tr**unken“ trennt Version 3 mitten in `kr` bzw. `tr`. Solche Konsonantenpaare
stehen im Deutschen oft gemeinsam am Anfang einer Silbe (Kraut, treten, Brot, Glas, ...). Eine Regel C kann das berücksichtigen:

- Steht an der Stelle ein typischer Silbenanfang (zum Beispiel `tr`, `kr`) **zusammen** in der Mitte, wird dort nicht getrennt.
- Folgt nach der Stelle ein typischer Silbenanfang und danach ein Vokal, darf **vor** ihm getrennt werden.

Regel C steht zwischen Regel 4 und Regel B, damit Regel B und Regel 2 weiterhin gewinnen. Nicht in die Liste gehören `st`, `sp`
und `pf`, denn nach heutiger Rechtschreibung trennt man „Kas-ten“, „Wes-pe“ und „Karp-fen“.

In [ ]:
ANFAENGE = ['bl', 'br', 'dr', 'fl', 'fr', 'gl', 'gr', 'kl', 'kn', 'kr', 'pl', 'pr', 'tr',
            'schl', 'schm', 'schn', 'schr', 'schw', 'zw']


def trennstellen_v4(teile):
    n = len(teile)
    stellen = []
    for p in range(1, n):
        erlaubt = False
        # Regel A
        if ist_konsonant(teile[p - 1]) and ist_konsonant(teile[p]) and p + 1 < n and ist_vokal(teile[p + 1]):
            erlaubt = True
        # Regel 4
        if ist_vokal(teile[p - 1]) and not (p + 1 < n and ist_konsonant(teile[p]) and ist_konsonant(teile[p + 1])):
            erlaubt = True
        # Regel C: einen typischen Silbenanfang nicht in der Mitte trennen, sondern davor
        if (teile[p - 1] + teile[p]).lower() in ANFAENGE:
            erlaubt = False
        if p + 2 < n and (teile[p] + teile[p + 1]).lower() in ANFAENGE and ist_vokal(teile[p + 2]):
            erlaubt = True
        # Regel B
        if not hat_vokal(teile[:p]) or not hat_vokal(teile[p:]):
            erlaubt = False
        # Regel 2
        if len(''.join(teile[:p])) == 1 or len(''.join(teile[p:])) == 1:
            erlaubt = False
        if erlaubt:
            stellen.append(p)
    return stellen


def trenne_v4(wort):
    teile = einheiten(wort)
    return markiere(teile, trennstellen_v4(teile))


def trenne(wort):
    return trenne_v4(wort)


print(trenne_v4('Sauerkraut'), trenne_v4('betreten'), trenne_v4('feuertrunken'))
bewerte('Version 4: Silbenanfänge')
fehler()

Sau-er-kraut be-tre-ten feu-er-trun-ken
Version 4: Silbenanfänge             richtig: 60  falsch:  3  fehlend:  2
Au-di-o-sig-nal                     richtig: Au-dio-sig-nal
an-gle                              richtig: ang-le
Hobby-te                            richtig: Hob-byte


Regel C behebt drei Fehler (Sauerkraut, betreten, feuertrunken) und verursacht einen neuen: „an-gle“ statt „ang-le“, weil `gl` in der Liste steht.
Insgesamt lohnt sie sich. Ob man `gl` aus der Liste nimmt, sollte man mit mehr Testwörtern entscheiden.

## Zusammenfassung der Bewertung

Zum Schluss alle Versionen auf einen Blick. Für die Dokumentation gehört außerdem dazu: wie gemessen wurde, welche Fehler bleiben
(zusammengesetzte Wörter, Fremdwörter, Wortspiele) und dass die Beispieltexte kurz sind. Eine Regel, die hier hilft, kann
bei anderen Wörtern schaden (zum Beispiel `ie` in „Ferien“ oder `gl` in „angle“). Besser wäre eine größere Wortliste.

In [ ]:
def trenne(wort):
    return trenne_lars(wort)
bewerte("Lars' Regeln")

def trenne(wort):
    return trenne_v1(wort)
bewerte('Version 1: Rand nie abtrennen')

def trenne(wort):
    return trenne_v2(wort)
bewerte('Version 2: Buchstabengruppen')

def trenne(wort):
    return trenne_v3(wort)
bewerte('Version 3: Duden-Regel und Vokal')

def trenne(wort):
    return trenne_v4(wort)
bewerte('Version 4: Silbenanfänge')

Lars' Regeln                         richtig: 45  falsch: 80  fehlend: 17
Version 1: Rand nie abtrennen        richtig: 45  falsch: 36  fehlend: 17
Version 2: Buchstabengruppen         richtig: 50  falsch: 15  fehlend: 12
Version 3: Duden-Regel und Vokal     richtig: 58  falsch:  5  fehlend:  4
Version 4: Silbenanfänge             richtig: 60  falsch:  3  fehlend:  2
